# Agente de IA para Análise de Vendas (Star Schema)

**Objetivo**: Agente conversacional que responde perguntas sobre vendas usando Gemini + SQL Server (modelo dimensional).

**Arquitetura**:
- Interface: Jupyter (desenvolvimento) → Streamlit (produção)
- LLM: Google Gemini
- Banco: SQL Server (Star Schema)
- Processamento: Pandas + Plotly

**Autor**:  Roberto souza 
**Data**: 22/09/2026

In [1]:
# Execute apenas uma vez

%pip install google-genai --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install --upgrade pydantic google-genai --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Imports e configuração

In [7]:
import os
import re
import json
from typing import Optional, Dict, Any, List
from datetime import datetime

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv
from google import genai
from google.genai import types
from sqlalchemy import create_engine, text

# Carrega as variáveis do arquivo .env
load_dotenv()

# Configurações
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
SQL_CONNECTION_STRING = os.getenv("SQL_CONNECTION_STRING")

# Cliente Gemini
client = genai.Client(api_key=GEMINI_API_KEY)

print("Chave Gemini carregada:", "Sim" if GEMINI_API_KEY else "Não")
print("Connection String carregada:", "Sim" if SQL_CONNECTION_STRING else "Não")

Chave Gemini carregada: Sim
Connection String carregada: Sim


## Constantes e Schema permitido (segurança)

In [8]:
# Schema permitido – nunca deixe o LLM inventar tabelas/colunas
ALLOWED_TABLES = {
    "D_CLIENTE": ["COD_CLIENTE", "NOME", "NOME_FANTASIA", "CLASSIFICACAO_CLIENTE", "CIDADE", "ESTADO", "UF"],
    "D_EMPRESA": ["COD_EMPRESA", "NOME", "NOME_FANTASIA"],
    "D_PRODUTO": ["COD_PRODUTO", "DESCRICAO", "DESCRICAO_REDUZIDA", "FAMILIA", "SECAO", "GRUPO", "SUB_GRUPO", "MARCA"],
    "D_VENDEDOR": ["COD_VENDEDOR", "NOME"],
    "F_VENDAS": ["N_DOC", "COD_EMPRESA", "COD_PRODUTO", "COD_VENDEDOR", "COD_CLIENTE", 
                 "MOVIMENTO", "QUANTIDADE", "VENDA_BRUTA", "DESCONTO_TOTAL", "VENDA_LIQUIDA"]
}

# Relações (para o prompt do Gemini)
SCHEMA_DESCRIPTION = """
Star Schema de Vendas:

DIMENSÕES:
- D_CLIENTE (COD_CLIENTE PK): NOME, NOME_FANTASIA, CLASSIFICACAO_CLIENTE, CIDADE, ESTADO, UF
- D_EMPRESA (COD_EMPRESA PK): NOME, NOME_FANTASIA
- D_PRODUTO (COD_PRODUTO PK): DESCRICAO, DESCRICAO_REDUZIDA, FAMILIA, SECAO, GRUPO, SUB_GRUPO, MARCA
- D_VENDEDOR (COD_VENDEDOR PK): NOME

FATO:
- F_VENDAS: N_DOC, COD_EMPRESA, COD_PRODUTO, COD_VENDEDOR, COD_CLIENTE, MOVIMENTO (date),
            QUANTIDADE (int), VENDA_BRUTA (money), DESCONTO_TOTAL (money), VENDA_LIQUIDA (money)

Joins típicos:
F_VENDAS.COD_CLIENTE = D_CLIENTE.COD_CLIENTE
F_VENDAS.COD_EMPRESA = D_EMPRESA.COD_EMPRESA
F_VENDAS.COD_PRODUTO = D_PRODUTO.COD_PRODUTO
F_VENDAS.COD_VENDEDOR = D_VENDEDOR.COD_VENDEDOR
"""

## Conexão com o SQL Server

In [13]:
import urllib.parse
from sqlalchemy import create_engine

def get_engine():
    connection_string = (
        "DRIVER={ODBC Driver 17 for SQL Server};"
        "SERVER=localhost\\SQLEXPRESS;"
        "DATABASE=DW;"
        "Trusted_Connection=yes;"
    )

    params = urllib.parse.quote_plus(connection_string)

    return create_engine(
        f"mssql+pyodbc:///?odbc_connect={params}"
    )

In [14]:
try:
    engine = get_engine()

    with engine.connect() as conn:
        print("✅ SQLAlchemy + PyODBC conectado ao SQL Server!")

except Exception as e:
    print(f"❌ Erro: {e}")

✅ SQLAlchemy + PyODBC conectado ao SQL Server!


In [17]:
import pandas as pd
from sqlalchemy import text

query = """
SELECT TOP 10 *
FROM D_CLIENTE
"""

with engine.connect() as conn:
    df = pd.read_sql(text(query), conn)

df

,COD_CLIENTE,NOME,NOME_FANTASIA,CLASSIFICACAO_CLIENTE,CIDADE,ESTADO,UF
0,0.0,Não Informado,Não Informado,Não Informada,Não Informada,Não Informado,ND
1,1.0,Cliente 1,Cliente 1,A Definir,Fortaleza,Ceara,CE
2,1001.0,Cliente 1001,Cliente 1001,Órgão Público,Fortaleza,Ceara,CE
3,1002.0,Cliente 1002,Cliente 1002,Órgão Privado,Fortaleza,Ceara,CE
4,1003.0,Cliente 1003,Cliente 1003,Não Informada,Fortaleza,Ceara,CE
5,1004.0,Cliente 1004,Cliente 1004,Órgão Público,Fortaleza,Ceara,CE
6,1005.0,Cliente 1005,Cliente 1005,Não Informada,Fortaleza,Ceara,CE
7,1006.0,Cliente 1006,Cliente 1006,Não Informada,Fortaleza,Ceara,CE
8,1007.0,Cliente 1007,Cliente 1007,Órgão Privado,Fortaleza,Ceara,CE
9,1008.0,Cliente 1008,Cliente 1008,Órgão Público,Fortaleza,Ceara,CE


## Funções de segurança SQL

In [18]:
def is_safe_sql(sql: str) -> bool:
    """Valida se a query é segura (apenas SELECT e tabelas/colunas permitidas)."""
    sql_upper = sql.upper().strip()
    
    # Bloqueia comandos perigosos
    forbidden = ["INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "TRUNCATE", 
                 "EXEC", "EXECUTE", "CREATE", "GRANT", "REVOKE", "--", ";--"]
    if any(cmd in sql_upper for cmd in forbidden):
        return False
    
    if not sql_upper.startswith("SELECT"):
        return False
    
    # Verifica tabelas (simples, mas eficaz)
    for table in re.findall(r'\bFROM\s+(\w+)|\bJOIN\s+(\w+)', sql_upper):
        t = table[0] or table[1]
        if t and t not in ALLOWED_TABLES:
            return False
    
    return True

def execute_safe_query(sql: str) -> pd.DataFrame:
    """Executa query apenas se for segura."""
    if not is_safe_sql(sql):
        raise ValueError("Query rejeitada por segurança. Apenas SELECT com tabelas permitidas.")
    
    engine = get_engine()
    with engine.connect() as conn:
        df = pd.read_sql(text(sql), conn)
    return df

## Prompt engineering + Gemini

In [19]:
def build_prompt(user_question: str) -> str:
    return f"""
Você é um analista de dados especializado em vendas. 
Seu único trabalho é converter a pergunta do usuário em uma query SQL Server válida e segura.

REGRAS OBRIGATÓRIAS:
1. Responda APENAS com um JSON no formato:
{{
  "intention": "descrição curta da intenção",
  "sql": "SELECT ...",
  "needs_chart": true/false,
  "chart_type": "bar|line|pie|table|none",
  "explanation": "explicação curta do que a query faz"
}}
2. Use APENAS as tabelas e colunas do schema abaixo.
3. Nunca use INSERT, UPDATE, DELETE, DROP, etc.
4. Prefira agregações (SUM, COUNT, AVG) quando fizer sentido.
5. Use aliases claros.
6. Para datas use MOVIMENTO.
7. Se a pergunta não puder ser respondida com o schema, retorne sql = null e explique.

SCHEMA:
{SCHEMA_DESCRIPTION}

PERGUNTA DO USUÁRIO:
{user_question}
"""

def ask_gemini(question: str) -> Dict[str, Any]:
    model = genai.GenerativeModel("gemini-1.5-flash")  # ou gemini-1.5-pro
    prompt = build_prompt(question)
    
    response = model.generate_content(
        prompt,
        generation_config={
            "temperature": 0.1,  # baixa para ser determinístico
            "response_mime_type": "application/json"
        }
    )
    
    try:
        return json.loads(response.text)
    except json.JSONDecodeError:
        # fallback simples
        return {
            "intention": "erro de parsing",
            "sql": None,
            "needs_chart": False,
            "chart_type": "none",
            "explanation": response.text
        }

## Orquestrador principal

In [45]:
def create_chart(df: pd.DataFrame, chart_type: str, title: str, category_threshold: int = 8):
    """Cria gráfico Plotly básico e inteligente.
    
    category_threshold: acima desse número de categorias, bar vira horizontal
    para melhor leitura dos rótulos.
    """
    if df.empty:
        return None

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    categorical_cols = df.select_dtypes(exclude="number").columns.tolist()

    if not numeric_cols:
        return None

    if categorical_cols:
        x_col = categorical_cols[0]
        y_candidates = numeric_cols
    else:
        x_col = numeric_cols[0]
        y_candidates = numeric_cols[1:] or numeric_cols

    y_col = next((c for c in y_candidates if c != x_col), numeric_cols[0])

    n_categories = df[x_col].nunique()

    if chart_type == "pie":
        fig = px.pie(df.sort_values(by=x_col), names=x_col, values=y_col, title=title)

    elif chart_type == "line":
        fig = px.line(df.sort_values(by=x_col), x=x_col, y=y_col, title=title)

    else:  # bar (default)
        if n_categories > category_threshold:
            # horizontal, ordenado do maior pro menor (fica de cima pra baixo na leitura)
            df_sorted = df.sort_values(by=y_col, ascending=True)
            fig = px.bar(
                df_sorted, x=y_col, y=x_col, orientation="h", title=title
            )
            # altura dinâmica: mais categorias = gráfico mais alto, senão fica espremido
            fig.update_layout(height=max(400, n_categories * 28))
        else:
            df_sorted = df.sort_values(by=x_col)
            fig = px.bar(df_sorted, x=x_col, y=y_col, title=title)

    fig.update_layout(template="plotly_white")
    return fig

## Interface interativa no Notebook

In [30]:
def chat(question: str):
    """Função amigável para testar o agente completo."""
    result = run_agent(question)

    if not result["success"]:
        print(f"❌ {result['message']}")
        return

    print(f"\n✅ {result['explanation']}")
    print(f"📊 Linhas retornadas: {result['row_count']}")

    display(result["data"].head(20))

    if result["fig"]:
        result["fig"].show()

    return result

In [39]:
import time
import random

def ask_gemini(question: str, max_retries: int = 3) -> Dict[str, Any]:
    prompt = build_prompt(question)

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-3.6-flash",
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                ),
            )
            return json.loads(response.text)

        except json.JSONDecodeError:
            return {
                "intention": "erro de parsing",
                "sql": None,
                "needs_chart": False,
                "chart_type": "none",
                "explanation": response.text,
            }

        except Exception as e:
            is_last_attempt = attempt == max_retries - 1
            if "503" in str(e) or "UNAVAILABLE" in str(e):
                if is_last_attempt:
                    return {
                        "intention": "erro de disponibilidade",
                        "sql": None,
                        "needs_chart": False,
                        "chart_type": "none",
                        "explanation": "O modelo Gemini está indisponível no momento (alta demanda). Tente novamente em instantes.",
                    }
                wait = (2 ** attempt) + random.uniform(0, 1)
                print(f"⚠️ 503 recebido, tentando novamente em {wait:.1f}s...")
                time.sleep(wait)
            else:
                raise  # erro diferente de 503 — deixa subir, não mascara

In [42]:
chat("Qual o total de vendas líquidas por mês em 2019?")

🔍 Pergunta: Qual o total de vendas líquidas por mês em 2019?


🧠 Intenção: Obter o total de vendas líquidas agrupado por mês no ano de 2019.
📝 SQL gerado:
SELECT MONTH(MOVIMENTO) AS MES, SUM(VENDA_LIQUIDA) AS TOTAL_VENDA_LIQUIDA FROM F_VENDAS WHERE YEAR(MOVIMENTO) = 2019 GROUP BY MONTH(MOVIMENTO) ORDER BY MES ASC

✅ A consulta filtra os dados de vendas para o ano de 2019 na tabela F_VENDAS, agrupa por mês através da coluna MOVIMENTO e calcula a soma da VENDA_LIQUIDA para cada mês.
📊 Linhas retornadas: 7


,MES,TOTAL_VENDA_LIQUIDA
0,6,3569425.45
1,7,8182328.77
2,8,8899305.86
3,9,9211356.64
4,10,8278676.27
5,11,9046397.68
6,12,9091037.17


{'success': True,
 'intention': 'Obter o total de vendas líquidas agrupado por mês no ano de 2019.',
 'explanation': 'A consulta filtra os dados de vendas para o ano de 2019 na tabela F_VENDAS, agrupa por mês através da coluna MOVIMENTO e calcula a soma da VENDA_LIQUIDA para cada mês.',
 'sql': 'SELECT MONTH(MOVIMENTO) AS MES, SUM(VENDA_LIQUIDA) AS TOTAL_VENDA_LIQUIDA FROM F_VENDAS WHERE YEAR(MOVIMENTO) = 2019 GROUP BY MONTH(MOVIMENTO) ORDER BY MES ASC',
 'data':    MES  TOTAL_VENDA_LIQUIDA
 0    6           3569425.45
 1    7           8182328.77
 2    8           8899305.86
 3    9           9211356.64
 4   10           8278676.27
 5   11           9046397.68
 6   12           9091037.17,
 'fig': Figure({
     'data': [{'hovertemplate': 'MES=%{x}<br>TOTAL_VENDA_LIQUIDA=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines',
               'name'

In [48]:
chat("Quais os 10 produtos mais vendidos em quantidade?")

🔍 Pergunta: Quais os 10 produtos mais vendidos em quantidade?
⚠️ 503 recebido, tentando novamente em 1.3s...
⚠️ 503 recebido, tentando novamente em 2.1s...


🧠 Intenção: Identificar os 10 produtos com maior quantidade vendida
📝 SQL gerado:
SELECT TOP 10 p.DESCRICAO AS PRODUTO, SUM(f.QUANTIDADE) AS TOTAL_QUANTIDADE FROM F_VENDAS f INNER JOIN D_PRODUTO p ON f.COD_PRODUTO = p.COD_PRODUTO GROUP BY p.DESCRICAO ORDER BY TOTAL_QUANTIDADE DESC

✅ A consulta realiza um agrupamento dos produtos somando a quantidade total vendida de cada um na tabela de fatos, retornando os 10 maiores resultados em ordem decrescente.
📊 Linhas retornadas: 10


,PRODUTO,TOTAL_QUANTIDADE
0,Produto 660,701087
1,Produto 33577,445737
2,Produto 34593,336697
3,Produto 33696,236043
4,Produto 2824,159644
5,Produto 3159,124440
6,Produto 33832,117746
7,Produto 34709,108237
8,Produto 2414,89747
9,Produto 35529,73700


{'success': True,
 'intention': 'Identificar os 10 produtos com maior quantidade vendida',
 'explanation': 'A consulta realiza um agrupamento dos produtos somando a quantidade total vendida de cada um na tabela de fatos, retornando os 10 maiores resultados em ordem decrescente.',
 'sql': 'SELECT TOP 10 p.DESCRICAO AS PRODUTO, SUM(f.QUANTIDADE) AS TOTAL_QUANTIDADE FROM F_VENDAS f INNER JOIN D_PRODUTO p ON f.COD_PRODUTO = p.COD_PRODUTO GROUP BY p.DESCRICAO ORDER BY TOTAL_QUANTIDADE DESC',
 'data':          PRODUTO  TOTAL_QUANTIDADE
 0    Produto 660            701087
 1  Produto 33577            445737
 2  Produto 34593            336697
 3  Produto 33696            236043
 4   Produto 2824            159644
 5   Produto 3159            124440
 6  Produto 33832            117746
 7  Produto 34709            108237
 8   Produto 2414             89747
 9  Produto 35529             73700,
 'fig': Figure({
     'data': [{'alignmentgroup': 'True',
               'hovertemplate': 'TOTAL_QUANTI

In [47]:
# Join com dimensão de cliente
chat("Quais os 5 clientes com maior faturamento líquido em 2020?")

🔍 Pergunta: Quais os 5 clientes com maior faturamento líquido em 2020?


🧠 Intenção: Identificar os 5 clientes com maior faturamento líquido no ano de 2020
📝 SQL gerado:
SELECT TOP 5 c.NOME AS CLIENTE, SUM(f.VENDA_LIQUIDA) AS FATURAMENTO_LIQUIDO FROM F_VENDAS f INNER JOIN D_CLIENTE c ON f.COD_CLIENTE = c.COD_CLIENTE WHERE YEAR(f.MOVIMENTO) = 2020 GROUP BY c.NOME ORDER BY FATURAMENTO_LIQUIDO DESC

✅ Agrupa as vendas pelo nome do cliente no ano de 2020, calcula a soma da VENDA_LIQUIDA e retorna os 5 maiores resultados em ordem decrescente.
📊 Linhas retornadas: 5


,CLIENTE,FATURAMENTO_LIQUIDO
0,Cliente 1876,9744749.59
1,Cliente 1132,5010801.61
2,Cliente 2105,3723220.00
3,Cliente 1274,3593063.40
4,Cliente 1885,3339585.84


{'success': True,
 'intention': 'Identificar os 5 clientes com maior faturamento líquido no ano de 2020',
 'explanation': 'Agrupa as vendas pelo nome do cliente no ano de 2020, calcula a soma da VENDA_LIQUIDA e retorna os 5 maiores resultados em ordem decrescente.',
 'sql': 'SELECT TOP 5 c.NOME AS CLIENTE, SUM(f.VENDA_LIQUIDA) AS FATURAMENTO_LIQUIDO FROM F_VENDAS f INNER JOIN D_CLIENTE c ON f.COD_CLIENTE = c.COD_CLIENTE WHERE YEAR(f.MOVIMENTO) = 2020 GROUP BY c.NOME ORDER BY FATURAMENTO_LIQUIDO DESC',
 'data':         CLIENTE  FATURAMENTO_LIQUIDO
 0  Cliente 1876           9744749.59
 1  Cliente 1132           5010801.61
 2  Cliente 2105           3723220.00
 3  Cliente 1274           3593063.40
 4  Cliente 1885           3339585.84,
 'fig': Figure({
     'data': [{'alignmentgroup': 'True',
               'hovertemplate': 'CLIENTE=%{x}<br>FATURAMENTO_LIQUIDO=%{y}<extra></extra>',
               'legendgroup': '',
               'marker': {'color': '#636efa', 'pattern': {'shape': ''}},


In [46]:
# Filtro por texto/categoria (testa geração de WHERE com string)
chat("Quais são 10 vendedores com maiores vendas ?")

🔍 Pergunta: Quais são 10 vendedores com maiores vendas ?


🧠 Intenção: Listar os 10 vendedores com o maior valor total em vendas líquidas
📝 SQL gerado:
SELECT TOP 10 v.NOME AS VENDEDOR, SUM(f.VENDA_LIQUIDA) AS TOTAL_VENDAS FROM F_VENDAS f INNER JOIN D_VENDEDOR v ON f.COD_VENDEDOR = v.COD_VENDEDOR GROUP BY v.NOME ORDER BY TOTAL_VENDAS DESC

✅ A consulta realiza a junção entre a fato de vendas e a dimensão vendedor, somando o total de vendas líquidas por vendedor e retornando os 10 maiores resultados.
📊 Linhas retornadas: 10


,VENDEDOR,TOTAL_VENDAS
0,Vendedor 16,45450285.90
1,Vendedor 4,21383278.75
2,Vendedor 3,14411746.64
3,Vendedor 1,14102553.58
4,Vendedor 15,4541177.49
5,Vendedor 5,4100392.96
6,Vendedor 17,3889677.70
7,Vendedor 20,3822690.12
8,Vendedor 18,3496210.86
9,Vendedor 2,3130316.18


{'success': True,
 'intention': 'Listar os 10 vendedores com o maior valor total em vendas líquidas',
 'explanation': 'A consulta realiza a junção entre a fato de vendas e a dimensão vendedor, somando o total de vendas líquidas por vendedor e retornando os 10 maiores resultados.',
 'sql': 'SELECT TOP 10 v.NOME AS VENDEDOR, SUM(f.VENDA_LIQUIDA) AS TOTAL_VENDAS FROM F_VENDAS f INNER JOIN D_VENDEDOR v ON f.COD_VENDEDOR = v.COD_VENDEDOR GROUP BY v.NOME ORDER BY TOTAL_VENDAS DESC',
 'data':       VENDEDOR  TOTAL_VENDAS
 0  Vendedor 16   45450285.90
 1   Vendedor 4   21383278.75
 2   Vendedor 3   14411746.64
 3   Vendedor 1   14102553.58
 4  Vendedor 15    4541177.49
 5   Vendedor 5    4100392.96
 6  Vendedor 17    3889677.70
 7  Vendedor 20    3822690.12
 8  Vendedor 18    3496210.86
 9   Vendedor 2    3130316.18,
 'fig': Figure({
     'data': [{'alignmentgroup': 'True',
               'hovertemplate': 'TOTAL_VENDAS=%{x}<br>VENDEDOR=%{y}<extra></extra>',
               'legendgroup': '',
  